# PV Layout System - Model Training & Testing Notebook
---
This notebook walks you through:
1. Generating synthetic training data
2. Training the obstacle classifier CNN
3. Evaluating model performance
4. Running the full PV layout pipeline on a test image
5. Visualising results and energy calculations

In [ ]:
# Setup & Imports
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from config import *
from roof_detector import RoofDetector
from obstacle_classifier import ObstacleClassifier, SyntheticObstacleGenerator, TORCH_AVAILABLE
from pv_optimizer import PVLayoutOptimizer
from energy_calculator import EnergyCalculator, quick_estimate, compare_regions
from visualization import LayoutVisualizer, ChartGenerator

print(f'PyTorch available: {TORCH_AVAILABLE}')
if TORCH_AVAILABLE:
    import torch
    print(f'CUDA available: {torch.cuda.is_available()}')
    print(f'Device: {torch.device("cuda" if torch.cuda.is_available() else "cpu")}')

## 1. Generate Synthetic Training Dataset
Since you're starting without a dataset, we'll generate synthetic obstacle images for initial training.

In [ ]:
# Generate synthetic dataset
generator = SyntheticObstacleGenerator()
generator.generate_dataset(
    output_dir=OBSTACLE_DATASET_DIR,
    samples_per_class=300,  # 300 images per class
    size=(128, 128)
)

# Preview generated samples
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, class_name in enumerate(CNN_CONFIG['class_names']):
    class_dir = os.path.join(OBSTACLE_DATASET_DIR, class_name)
    files = os.listdir(class_dir)
    if files:
        img = cv2.imread(os.path.join(class_dir, files[0]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i].imshow(img)
        axes[i].set_title(class_name)
        axes[i].axis('off')
plt.suptitle('Synthetic Training Samples', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Train the Obstacle Classifier CNN

In [ ]:
# Initialize classifier
classifier = ObstacleClassifier()

# Prepare data loaders
train_loader, val_loader, test_loader = classifier.prepare_datasets(OBSTACLE_DATASET_DIR)

# Train the model
history = classifier.train(
    train_loader, val_loader,
    epochs=30,  # Adjust as needed
    lr=0.001
)

print('\nTraining complete!')

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs, history['train_loss'], 'b-', label='Train')
ax1.plot(epochs, history['val_loss'], 'r-', label='Validation')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs, history['train_acc'], 'b-', label='Train')
ax2.plot(epochs, history['val_acc'], 'r-', label='Validation')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
metrics = classifier.evaluate(test_loader)
print(f'Overall Test Accuracy: {metrics["overall_accuracy"]:.2%}')
print(f'\nPer-class accuracy:')
for cls, acc in metrics['class_accuracy'].items():
    print(f'  {cls}: {acc:.2%}')

## 3. Test the Full Pipeline
Create a synthetic roof image to demonstrate the complete workflow.

In [ ]:
# Create a synthetic roof image for testing
def create_test_roof_image(width=800, height=600):
    """Generate a test roof image with obstacles."""
    img = np.full((height, width, 3), (180, 170, 160), dtype=np.uint8)
    
    # Draw roof area (lighter polygon)
    roof_pts = np.array([
        [100, 80], [700, 80], [720, 520], [80, 520]
    ], dtype=np.int32)
    cv2.fillPoly(img, [roof_pts], (200, 195, 185))
    cv2.polylines(img, [roof_pts], True, (120, 115, 110), 2)
    
    # Add water tank (darker rectangle)
    cv2.rectangle(img, (500, 100), (600, 180), (100, 95, 90), -1)
    cv2.rectangle(img, (500, 100), (600, 180), (80, 75, 70), 2)
    
    # Add vent (circle)
    cv2.circle(img, (200, 400), 30, (90, 85, 80), -1)
    cv2.circle(img, (200, 400), 30, (70, 65, 60), 2)
    
    # Add texture
    noise = np.random.randint(-8, 8, img.shape, dtype=np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    
    return img

test_image = create_test_roof_image()
plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB))
plt.title('Synthetic Test Roof Image')
plt.axis('off')
plt.show()

In [ ]:
# Step 1: Detect roof boundary
detector = RoofDetector()
detector.load_image_from_array(test_image)

# Option A: Auto-detect
try:
    roof_contour = detector.detect_roof_boundary(method='auto')
    print(f'Roof detected with {len(roof_contour)} vertices')
except:
    # Option B: Manual definition (for synthetic image)
    roof_contour = detector.manually_define_roof(
        [(100, 80), (700, 80), (720, 520), (80, 520)]
    )
    print('Roof manually defined')

# Set scale: assume roof is 20m x 15m
detector.set_scale_from_roof_dims(20.0, 15.0)
print(f'Scale: {detector.scale_m_per_px:.4f} m/px')
print(f'Total roof area: {detector.get_roof_area_m2():.1f} m²')
print(f'Usable area: {detector.get_usable_area_m2():.1f} m²')

In [ ]:
# Step 2: Detect and classify obstacles
candidates = detector.detect_potential_obstacles()
print(f'Found {len(candidates)} potential obstacles')

# Classify with CNN (if trained)
if classifier.is_trained:
    candidates = classifier.classify_batch(candidates)
    for obs in candidates:
        print(f'  Obstacle {obs["id"]}: {obs["label"]} (conf: {obs["confidence"]:.2%})')

# Remove obstacles from usable area
obstacle_contours = [obs['contour'] for obs in candidates if obs.get('label') != 'none']
if obstacle_contours:
    detector.remove_obstacles_from_mask(obstacle_contours)
    print(f'Usable area after obstacles: {detector.get_usable_area_m2():.1f} m²')

In [ ]:
# Step 3: Optimise panel layout
optimizer = PVLayoutOptimizer()
panels = optimizer.optimize_layout(
    usable_area_mask=detector.usable_area_mask,
    scale_m_per_px=detector.scale_m_per_px,
    orientation='auto',
    tilt_deg=10,
    add_walkway=True
)

summary = optimizer.get_layout_summary()
print('\n=== LAYOUT SUMMARY ===')
for key, val in summary.items():
    print(f'  {key}: {val}')

inverter = optimizer.get_inverter_recommendation()
print('\n=== INVERTER RECOMMENDATION ===')
for key, val in inverter.items():
    print(f'  {key}: {val}')

In [ ]:
# Step 4: Calculate energy yield
calc = EnergyCalculator(region='Kuala_Lumpur')
calc.set_system(n_panels=len(panels), tilt_deg=10)

yield_data = calc.calculate_annual_yield()
print('\n=== ENERGY YIELD ===')
for key, val in yield_data.items():
    if key != 'monthly_yield_kwh':
        print(f'  {key}: {val}')

financial = calc.calculate_financial_returns(yield_data['net_annual_yield_kwh'])
print('\n=== FINANCIAL ANALYSIS ===')
print(f'  System Cost: MYR {financial["total_system_cost_myr"]:,.0f}')
print(f'  Year 1 Savings: MYR {financial["year1_savings_myr"]:,.0f}')
print(f'  Payback Period: {financial["simple_payback_years"]} years')
print(f'  25-year ROI: {financial["roi_pct"]}%')
print(f'  LCOE: MYR {financial["lcoe_myr_kwh"]}/kWh')

In [ ]:
# Step 5: Visualise results
result_image = LayoutVisualizer.draw_layout_overlay(
    image=test_image,
    panels=panels,
    scale_m_per_px=detector.scale_m_per_px,
    roof_contour=detector.roof_contour,
    obstacles=candidates,
    usable_mask=detector.usable_area_mask,
    show_strings=True,
    alpha=0.5
)

# Add info panel
final_image = LayoutVisualizer.draw_info_panel(
    result_image, summary, yield_data
)

plt.figure(figsize=(16, 8))
plt.imshow(cv2.cvtColor(final_image, cv2.COLOR_BGR2RGB))
plt.title('Optimised PV Layout', fontsize=16)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Monthly yield chart
months = list(yield_data['monthly_yield_kwh'].keys())
values = list(yield_data['monthly_yield_kwh'].values())

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(months, values, color='#2196F3')
ax.set_ylabel('Energy (kWh)')
ax.set_title(f'Estimated Monthly Yield - {yield_data["net_annual_yield_kwh"]:,.0f} kWh/year')
ax.grid(axis='y', alpha=0.3)
for i, v in enumerate(values):
    ax.text(i, v + 10, f'{v:.0f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Financial payback chart
years = [d['year'] for d in financial['yearly_breakdown']]
cumulative = [d['cumulative_profit_myr'] for d in financial['yearly_breakdown']]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(years, cumulative, 'b-o', markersize=3)
ax.axhline(y=0, color='r', linestyle='--')
ax.fill_between(years, cumulative, 0, where=[c>=0 for c in cumulative], alpha=0.3, color='green')
ax.fill_between(years, cumulative, 0, where=[c<0 for c in cumulative], alpha=0.3, color='red')
ax.set_xlabel('Year')
ax.set_ylabel('Cumulative Profit (MYR)')
ax.set_title('25-Year Financial Projection')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Regional comparison
region_data = compare_regions(n_panels=len(panels))

regions = list(region_data.keys())
yields_r = [region_data[r]['annual_yield_kwh'] for r in regions]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(regions, yields_r, color='#FF9800')
ax.set_xlabel('Annual Yield (kWh)')
ax.set_title('Energy Yield by Malaysian Region')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. How to Add Your Own Training Data

To improve the model with real roof obstacle images:

1. **Collect images**: Crop obstacle regions from real roof photos
2. **Organise into folders**:
   ```
   dataset/obstacles/
       water_tank/   <- Images of water tanks
       parapet/      <- Images of parapet walls
       vent/         <- Images of vents/exhausts
       ac_unit/      <- Images of AC units
       none/         <- Images of empty roof (no obstacle)
   ```
3. **Re-run training** with `train_model.py --mode both`

Tips:
- Aim for 50+ images per class minimum
- Include varied angles, lighting, and zoom levels
- The synthetic data supplements your real data